# Task 1 — Preparazione del dataset (Bank Marketing)



## 0. Import e riproducibilità

Importiamo le librerie di base (`pandas` e `numpy`) e fissiamo un **seme casuale** (`SEED = 10`) in modo da rendere i risultati **riproducibili**, cioè a
garantire che il campionamento casuale delle istanze per `manuale.csv` produca sempre lo stesso
sottoinsieme a ogni esecuzione.

In [1]:
import pandas as pd
import numpy as np

In [2]:
SEED = 10
np.random.seed(SEED)

## 1. Caricamento del dataset

Carichiamo il dataset originale **Bank Marketing** dalla cartella `data/raw/` e specifichiamo con `sep=";"` che le colonne della tabella sono separate dal carattere `;`.

In [3]:
PATH = "../data/raw/bank-additional-full.csv"
dataFrame = pd.read_csv(PATH, sep=";")

Ci facciamo restituire la **dimensione** del dataset e alcune istanze (in questo caso le prime 10).

In [4]:
print(f"Il data set analizzato ha: {dataFrame.shape[0]} istanze e {dataFrame.shape[1]} attributi.")

Il data set analizzato ha: 41188 istanze e 21 attributi.


In [5]:
dataFrame.head(10)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
5,45,services,married,basic.9y,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
6,59,admin.,married,professional.course,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
7,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
8,24,technician,single,professional.course,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
9,25,services,single,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Controllo qualità di base

## 2a. Rimozione dei duplicati iniziali

Per scelta progettuale, si è deciso di rimuovere i campioni duplicati presenti nel dataset iniziale, pari a 12 istanze. In seguito alle operazioni di preprocessing sono emersi ulteriori duplicati; tuttavia, essi non costituiscono un problema ai fini dell'analisi, poiché rappresentano osservazioni perfettamente coerenti con il dominio applicativo e non alterano la distribuzione dei dati. Di conseguenza, i modelli possono essere addestrati anche in presenza di tali campioni, in quanto la loro presenza non determina effetti negativi sulle prestazioni né introduce distorsioni significative nei risultati.


In [6]:
prima = len(dataFrame)
dataFrame = dataFrame.drop_duplicates().reset_index(drop=True)
dopo = len(dataFrame)
print(prima-dopo , "istanze eliminate")

12 istanze eliminate


### 2b. Distribuzione della classe

Esaminiamo la distribuzione della variabile target `y`. Questo passaggio è cruciale perché
ci permette di capire se il dataset e sbilanciato oppure no. I
clienti che sottoscrivono il deposito (`yes`) sono una netta minoranza rispetto a chi rifiuta
(`no`). Le scelte che verranno prese successivamente dipenderanno da questa distribuzione.

In [7]:
dataFrame["y"].value_counts()

y
no     36537
yes     4639
Name: count, dtype: int64

In [8]:
percentualeYes = round(dataFrame["y"].value_counts(normalize=True)["yes"] * 100, 2)
print(f"Proporzione 'yes': {percentualeYes}%")

Proporzione 'yes': 11.27%


### 2c. Valori `unknown`

Nel dataset i valori mancanti non sono codificati come `NaN`, ma con la stringa esplicita
**`unknown`** in diversi attributi nominali.


In [9]:
# Individuazione dei valori 'unknown' nelle colonne nominali).
nominali = dataFrame.select_dtypes(include=["object", "string"]).columns

unknown_count = (dataFrame[nominali] == "unknown").sum()     # conteggio per colonna
unknown_count = unknown_count[unknown_count > 0]        # solo colonne con unknown

riepilogo_unknown = pd.DataFrame({
    "unknown": unknown_count,
    "%": (unknown_count / len(dataFrame) * 100).round(1),
})
print(riepilogo_unknown)

           unknown     %
job            330   0.8
marital         80   0.2
education     1730   4.2
default       8596  20.9
housing        990   2.4
loan           990   2.4


Trattamento dei valori unknown, mettiamo la moda per i nominali, chiaramente i numerici non possono avere unknown altrimenti non sarebbero numerici.

In [10]:

dataFrame = dataFrame.replace("unknown", np.nan)        # sostituzione unknown con NaN
nominali = dataFrame.select_dtypes(include=["object", "string"]).columns
dataFrame[nominali] = dataFrame[nominali].fillna(dataFrame[nominali].mode().iloc[0])  # sostituzione NaN con la moda, per valori nominali

# controlliamo se la sostituzione è andata a buon fine
NaN_count = (dataFrame[nominali] == np.nan).sum()     
NaN_count = NaN_count[NaN_count > 0]

riepilogo_NaN = pd.DataFrame({
    "NaN": NaN_count,
    "%": (NaN_count / len(dataFrame) * 100).round(1),
})
print(riepilogo_NaN)

Empty DataFrame
Columns: [NaN, %]
Index: []


## 2d. Trattamento della feature duration.
Poiché la feature è direttamente legata al target, si è deciso in fase progettuale di eliminarla.

In [11]:
# Rimozione colonna "duration"
dataFrame = dataFrame.drop(columns = ["duration"])
dataFrame.shape

(41176, 20)

### 2.e Duplicati
Come anticipato, si è scelto di eliminare i duplicati presenti nella versione iniziale del dataset, così da mostrare in modo trasparente come le successive operazioni di pulizia possano generare nuovi duplicati. Questi duplicati “post‑trattamento” non rappresentano tuttavia un problema: riflettono semplicemente osservazioni identiche ma valide e non alterano né l’analisi statistica né l’addestramento dei modelli, che possono operare correttamente anche in loro presenza.

In [12]:
print("Righe duplicate dopo il processamento:", dataFrame.duplicated().sum())

Righe duplicate dopo il processamento: 2097


## 3. Codifica della classe

Trasformiamo la variabile target da stringa a intero: **`no` → 0** e **`yes` → 1**. La classe
positiva di interesse (la sottoscrizione, l'evento "raro" che vogliamo prevedere) è quindi
codificata come **`1`**. Questa codifica binaria è il formato atteso dai classificatori e
rende immediato il calcolo di metriche come precision e recall sulla classe positiva.

In [13]:
dataFrame["y"] = dataFrame["y"].map({"no": 0, "yes": 1})
print("Classe y codificata: no->0, yes->1 (yes = classe positiva)")

Classe y codificata: no->0, yes->1 (yes = classe positiva)


In [14]:
dataFrame["y"].value_counts()

y
0    36537
1     4639
Name: count, dtype: int64

## 4. Estrazione dei due file
### 4a. `training.csv`

In [15]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [16]:
training = dataFrame.copy()
training.to_csv("../data/processed/training.csv",sep=";", index=False)
print(f"Salvato training.csv: {training.shape[0]} istanze x {training.shape[1]} attributi")

Salvato training.csv: 41176 istanze x 20 attributi


### 4b. `manuale.csv`

Estraiamo ora il file `manuale.csv`.
Costruiamo un insieme di **12 istanze deliberatamente bilanciate**: 6 della classe `yes` e 6
della classe `no`, campionate casualmente (con il seme fissato) e poi mescolate così che le
classi non risultino raggruppate.

**Due scelte di progetto importanti:**

- **Bilanciamento.** A differenza del dataset reale (fortemente sbilanciato), `manuale.csv` è
  bilanciato di proposito: serve a costruire e illustrare i classificatori a mano (Task 2) in
  un contesto pulito, dove entrambe le classi sono ugualmente rappresentate.
- **Sottoinsieme di feature.** Selezioniamo solo **9 attributi** (2 numerici e 7 nominali) tra
  i 20 disponibili. Sono attributi comprensibili e gestibili nel calcolo manuale: l'obiettivo
  del Task 2 non è la performance, ma mostrare *passo per passo* come funzionano gli algoritmi.


In [17]:
feature_manuale = [
    "age", "campaign",                # numerici
    "job", "marital", "education",    # nominali
    "housing", "loan", "contact", "poutcome",
    "y",                              # target
]

In [18]:
# 6 istanze 'yes' e 6 'no' -> set bilanciato (12 istanze)
yes_rows = dataFrame[dataFrame["y"] == 1].sample(n=6, random_state=SEED)
no_rows  = dataFrame[dataFrame["y"] == 0].sample(n=6, random_state=SEED)

In [19]:
manuale = pd.concat([yes_rows, no_rows])[feature_manuale]
# Mescoliamo l'ordine cosi' le classi non sono raggruppate
manuale = manuale.sample(frac=1, random_state=SEED).reset_index(drop=True)

In [20]:
manuale.to_csv("../data/processed/manuale.csv",sep=";", index=False)
print(f"Salvato manuale.csv: {manuale.shape[0]} istanze x {manuale.shape[1]} attributi")

Salvato manuale.csv: 12 istanze x 10 attributi


In [21]:
manuale

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,university.degree,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,university.degree,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


## Riepilogo

Abbiamo prodotto:
- **`training.csv`** — dataset pulito completo, pronto per EDA e addestramento
- **`manuale.csv`** — 12 istanze bilanciate con attributi adatti ai calcoli a mano

**Prossimo passo (Task 2):** definire a mano i due classificatori (Naïve Bayes e
1R) su `manuale.csv`, illustrare i passi per adattarli ai dati, implementarli in
Python e valutarne le prestazioni.
